In [1]:
import os
import re
import json
import glob
import asyncio
import requests
from pathlib import Path
from typing import TypedDict, List, Dict, Any, Optional, Iterable
from datetime import datetime
from collections import defaultdict
from tabulate import tabulate
from rank_bm25 import BM25Okapi
from time import perf_counter  # <<< ADD

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=False)

import chromadb  # pure chroma
from chromadb.api.types import EmbeddingFunction, Documents, Embeddings
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama  # hanya LLM-nya aja

# MCP client
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport

# --- TOTAL TIMING HELPERS (ADD) ---
def _now() -> float:
    return perf_counter()

def _elapsed_ms(t0: float) -> int:
    return int((perf_counter() - t0) * 1000)

def _log_total(label: str, t0: float):
    ms = _elapsed_ms(t0)
    print(f"[TOTAL] {label}: {ms} ms (~{ms/1000:.3f}s)")
# ----------------------------------

# =========================
# === CONFIG (ENV) ========
# =========================
JSON_INPUT       = os.getenv("EXPORT_JSONL_OUTPUT", "./datahub_metadata/*.jsonl").strip()
CHROMA_DIR       = os.getenv("CHROMA_DIR", "./chroma_db")
COLLECTION_NAME  = os.getenv("COLLECTION_NAME", "classicmodels")
OLLAMA_BASE_URL  = os.getenv("OLLAMA_URL", "http://localhost:11434")
CHAT_MODEL       = os.getenv("CHAT_MODEL", "HridaAI/hrida-t2sql:q8_0")
TOP_K            = int(os.getenv("TOP_K", "5"))

MICRO_PROBE = ["id", "date", "status"]

SPARSE_BONUS     = float(os.getenv("SPARSE_BONUS", "0.75"))
NAME_QUERY_BONUS = float(os.getenv("NAME_QUERY_BONUS", "0.3"))
ID_QUERY_BONUS   = float(os.getenv("ID_QUERY_BONUS", "0.4"))

# >>> SPEED: batasi berapa banyak dokumen yang masuk ke context LLM
MAX_CTX_DOCS = int(os.getenv("MAX_CTX_DOCS", "12"))

# bisa dimatiin kalau mau lebih silent
SPARSE_DEBUG   = bool(int(os.getenv("SPARSE_DEBUG", "0")))
RETRIEVE_DEBUG = bool(int(os.getenv("RETRIEVE_DEBUG", "1")))

COLUMN_PRIORITY_CONFIG: List[Dict[str, Any]] = [
    {
        "id": "customers.primary_name",
        "tables": ["customers"],
        "role_any": ["name"],
        "alias_any": [
            "customer name",
            "nama customer",
            "nama pelanggan",
            "company name",
            "nama perusahaan",
            "account name",
        ],
        "column_contains": ["customername"],
        "query_terms_any": ["name", "nama"],
        "query_terms_also_any": ["customer", "customers", "pelanggan"],
        "boost": 0.4,
    },
]

MCP_BASE    = os.getenv("MCP_BASE", "http://mcp_server:8000/mcp")
MCP_TIMEOUT = int(os.getenv("MCP_CLIENT_TIMEOUT", "60"))

SQL_DEBUG = int(os.getenv("SQL_DEBUG", "1"))
def _dbg_print_two(label: str, sql: str):
    if SQL_DEBUG:
        print(f"\n{label}")
        print(sql)

# =========================
# === RAG (STATE) =========
# =========================
class RAGState(TypedDict):
    jsonl_paths: List[str]
    question: str
    added: int
    updated: int
    skipped: int
    context: str
    answer: str
    draft_sql: str
    final_sql: str

# =========================
# === UTIL ================
# =========================
def resolve_jsonl_paths(spec: str) -> List[str]:
    paths: List[str] = []
    if not spec:
        return paths
    p = Path(spec)
    if any(ch in spec for ch in "*?[]"):
        paths.extend(glob.glob(spec))
    elif p.is_dir():
        paths.extend(str(x) for x in p.glob("*.jsonl"))
    elif p.is_file():
        paths.append(str(p))
    else:
        paths.extend(glob.glob(spec))
    out, seen = [], set()
    for x in paths:
        xp = str(Path(x))
        if xp.lower().endswith(".jsonl") and xp not in seen:
            seen.add(xp)
            out.append(xp)
    return out

class OllamaEmbeddingFunction(EmbeddingFunction):
    def __init__(self, base_url: str, model: str = "bge-m3:latest", timeout: int = 60):
        self.base_url = base_url.rstrip("/")
        self.model = model
        self.timeout = int(timeout)

    # Required: __call__(Documents) -> Embeddings
    def __call__(self, texts: Documents) -> Embeddings:
        out = []
        for t in texts:
            resp = requests.post(
                f"{self.base_url}/api/embeddings",
                json={"model": self.model, "prompt": t},
                timeout=self.timeout,
            )
            resp.raise_for_status()
            out.append(resp.json()["embedding"])
        return out

    # Required: class-level name()
    @classmethod
    def name(cls) -> str:
        # stable type name; model details belong in config
        return "ollama"

    # Required: JSON-serializable config (no callables)
    def get_config(self) -> dict:
        return {
            "base_url": self.base_url,
            "model": self.model,
            "timeout": self.timeout,
        }

    # Required: rebuild from config
    @classmethod
    def build_from_config(cls, config: dict) -> "OllamaEmbeddingFunction":
        return cls(
            base_url=str(config.get("base_url", "http://localhost:11434")),
            model=str(config.get("model", "bge-m3:latest")),
            timeout=int(config.get("timeout", 60)),
        )

    # new: allow Chroma to serialize EF config
    def get_config(self) -> Dict[str, Any]:
        return {
            "name": self.name(),
            "base_url": self.base_url,
            "model": self.model,
            "timeout": self.timeout,
        }

def build_vector_store():
    os.makedirs(CHROMA_DIR, exist_ok=True)
    client = chromadb.PersistentClient(path=CHROMA_DIR)
    emb_fn = OllamaEmbeddingFunction(
        base_url=OLLAMA_BASE_URL,
        model="bge-m3:latest",
    )
    col = client.get_or_create_collection(
        COLLECTION_NAME,
        embedding_function=emb_fn,
    )
    return col

def read_jsonl(path: str) -> Iterable[tuple[str, Optional[Dict[str, Any]]]]:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            try:
                obj = json.loads(s)
            except Exception:
                obj = None
            yield s, obj

# --- Chroma metadata sanitizer (PENTING) ---
def _sanitize_metadata(meta: Dict[str, Any]) -> Dict[str, Any]:
    """
    Chroma hanya menerima value metadata bertipe: str|int|float|bool|None.
    Di sini kita konversi list/dict -> JSON string; tipe lain -> str().
    """
    out: Dict[str, Any] = {}
    for k, v in (meta or {}).items():
        if isinstance(v, (str, int, float, bool)) or v is None:
            out[k] = v
        elif isinstance(v, (list, dict)):
            out[k] = json.dumps(v, ensure_ascii=False)
        else:
            out[k] = str(v)
    return out

# =========================
# === SPARSE (BM25) =======
# =========================
_BM25_CACHE: dict[str, Any] = {}
_STOPWORDS = {
    "the","a","an","to","of","and","or","for","on","in","at","by",
    "yang","dan","atau","ke","dari","di","itu","ini","with"
}

def _normalize_text(txt: str) -> str:
    txt = txt.lower()
    txt = re.sub(r"(\d)[,\.](\d{3})(\b|$)", r"\1\2", txt)
    txt = re.sub(r"(\w)'s\b", r"\1", txt)
    txt = txt.replace("'", "")
    txt = re.sub(r"[^a-z0-9_]+", " ", txt)
    return txt.strip()

def _tokenize(txt: str) -> list[str]:
    norm = _normalize_text(txt)
    toks = [t for t in norm.split() if t and t not in _STOPWORDS]
    return toks

def _collect_jsonl_docs(paths: list[str]) -> list[dict[str, Any]]:
    docs: list[dict[str, Any]] = []
    for path in paths:
        p = Path(path)
        if not p.exists():
            continue
        for _, obj in read_jsonl(path):
            if not isinstance(obj, dict):
                continue
            field_urn = obj.get("field_urn") or obj.get("urn")
            if not field_urn:
                continue

            text_bits: list[str] = []
            for key in ("database", "table", "column", "description", "database_description"):
                v = obj.get(key)
                if isinstance(v, str):
                    text_bits.append(v)

            for key in ("tags_alias", "tags_role"):
                v = obj.get(key)
                if isinstance(v, list):
                    text_bits.extend(str(x) for x in v)
                elif isinstance(v, str):
                    text_bits.append(v)

            full_txt = " ".join(text_bits)
            docs.append({
                "id": field_urn,
                "text": full_txt,
            })
    return docs

def _build_bm25_index(paths: list[str]) -> dict[str, Any]:
    key = tuple(sorted(paths))
    if key in _BM25_CACHE:
        return _BM25_CACHE[key]

    raw_docs = _collect_jsonl_docs(paths)
    tokenized_corpus = [_tokenize(d["text"]) for d in raw_docs]

    if tokenized_corpus:
        bm25 = BM25Okapi(tokenized_corpus)
    else:
        bm25 = None

    idx = {
        "docs": raw_docs,
        "tokens": tokenized_corpus,
        "bm25": bm25,
    }
    _BM25_CACHE[key] = idx
    return idx

def build_sparse_query_from_question(q: str) -> str:
    if not q:
        return ""
    s = q.lower()
    s = re.sub(r"(\d)[,\.](\d)", r"\1\2", s)
    s = re.sub(r"[^0-9a-z_]+", " ", s)
    s = re.sub(r"\s{2,}", " ", s).strip()
    return s

SPARSE_MAX_HITS = int(os.getenv("SPARSE_MAX_HITS", "10"))

def _singularize_token(tok: str) -> str:
    if tok.endswith("ies"):
        return tok[:-3] + "y"
    if tok.endswith("s") and len(tok) > 3:
        return tok[:-1]
    return tok

INSTRUCTION_TOKENS = {"list", "show", "tampilkan", "all", "display"}

def _tokenize_bm25_query(q: str) -> list[str]:
    norm_q = _normalize_text(q)
    toks = [t for t in norm_q.split() if t and t not in _STOPWORDS]
    toks = [t for t in toks if t not in INSTRUCTION_TOKENS]
    return [_singularize_token(t) for t in toks]

def jsonl_sparse_search(q_raw: str, paths: List[str], max_hits: int = SPARSE_MAX_HITS) -> list[str]:
    if not q_raw or not paths:
        if SPARSE_DEBUG:
            print("[BM25] empty query or paths -> no sparse hits")
        return []

    idx = _build_bm25_index(paths)
    bm25 = idx.get("bm25")
    raw_docs = idx.get("docs") or []
    if not bm25 or not raw_docs:
        if SPARSE_DEBUG:
            print("[BM25] index is empty -> no sparse hits")
        return []

    query_toks = _tokenize_bm25_query(q_raw)

    if SPARSE_DEBUG:
        print("[BM25] original query  :", repr(q_raw))
        print("[BM25] tokens          :", query_toks)

    if not query_toks:
        return []

    scores = bm25.get_scores(query_toks)

    boost = {}
    if any(t in ("country", "countries") for t in query_toks):
        for i, doc in enumerate(raw_docs):
            if "country" in doc["text"].lower():
                boost[i] = 2.0

    wants_customer_name = (
        any(t in ("customer", "customers", "pelanggan") for t in query_toks)
        and not any(t in ("id", "code", "customernumber", "customer_number", "customerid") for t in query_toks)
    )
    if wants_customer_name:
        for i, doc in enumerate(raw_docs):
            txt = doc["text"].lower()
            if "customername" in txt or "customer name" in txt:
                boost[i] = boost.get(i, 0.0) + 3.0
            elif "customernumber" in txt or "customer number" in txt:
                boost[i] = boost.get(i, 0.0) - 1.5

    scored = []
    for i, (doc, score) in enumerate(zip(raw_docs, scores)):
        score = score + boost.get(i, 0.0)
        scored.append((doc, score))

    scored.sort(key=lambda x: x[1], reverse=True)
    top = scored[:max_hits]
    bm25_ids = [doc["id"] for doc, _ in top]

    ql = q_raw.lower()
    need_moneyish = any(k in ql for k in ["credit limit", "creditlimit", "credit", "limit"])
    if need_moneyish and len(bm25_ids) < max_hits:
        extra: list[str] = []
        for path in paths:
            p = Path(path)
            if not p.exists():
                continue
            for _, obj in read_jsonl(path):
                if not isinstance(obj, dict):
                    continue
                field_urn = obj.get("field_urn") or obj.get("urn")
                if not field_urn:
                    continue
                col = (obj.get("column") or "").lower()
                desc = (obj.get("description") or "").lower()
                if ("credit" in col or "limit" in col or "credit" in desc or "limit" in desc):
                    if field_urn not in bm25_ids and field_urn not in extra:
                        extra.append(field_urn)
                if len(bm25_ids) + len(extra) >= max_hits:
                    break
        bm25_ids = bm25_ids + extra

    if SPARSE_DEBUG:
        print("[BM25] top field_urns  :")
        if bm25_ids:
            for u in bm25_ids:
                print("  -", u)
        else:
            print("  (no hits)")

    return bm25_ids[:max_hits]

def _doc_matches_sparse(md: Dict[str, Any], sparse_field_urns: list[str]) -> bool:
    if not sparse_field_urns or not md:
        return False
    field_urn = md.get("field_urn") or ""
    if not field_urn:
        return False
    return field_urn in set(sparse_field_urns)

# =========================
# === DENSE PROBE ========
# =========================
def build_dynamic_probe(question: str, max_tokens: int = 28) -> str:
    q = (question or "").lower()

    def _mentions_id_like(text: str) -> bool:
        if re.search(r"\b(id|number|nomor|no|code|kode)\b", text):
            return True
        if "customer number" in text or "customer_number" in text:
            return True
        return False

    identity_only = (
        any(k in q for k in ["name", "nama"])
        and not any(k in q for k in [
            "sum", "total", "jumlah", "revenue", "sales", "omset", "gross", "net",
            "count", "berapa banyak", "how many", "hitung",
            "avg", "average", "mean", "rata", "rata-rata",
            "today", "yesterday", "last", "this", "bulan", "minggu", "tahun", "quarter",
            "month", "week", "year", "202", "201", "tanggal", "hari", "mingguan", "bulanan"
        ])
    )

    base: List[str] = [] if identity_only else MICRO_PROBE[:]

    if any(k in q for k in ["avg", "average", "mean", "rata", "rata-rata"]):
        base += ["avg", "mean", "average", "metric", "measure", "price"]
    if any(k in q for k in ["sum", "total", "jumlah", "revenue", "sales", "omset", "gross", "net"]):
        base += ["sum", "total", "amount", "revenue", "sales", "price", "gross", "net"]
    if any(k in q for k in ["count", "berapa banyak", "how many", "hitung"]):
        base += ["count", "qty", "quantity", "number"]
    if any(k in q for k in [
        "today", "yesterday", "last", "this", "bulan", "minggu", "tahun", "quarter",
        "month", "week", "year", "202", "201", "tanggal", "hari", "mingguan", "bulanan"
    ]):
        base += ["date", "timestamp", "month", "week", "quarter", "year", "created", "updated", "order_date"]

    mentions_customer = any(k in q for k in ["customer", "customers", "pelanggan"])
    if mentions_customer:
        base += ["customer", "customer name"]
        if not _mentions_id_like(q):
            base += ["customerName"]

    if any(k in q for k in ["country", "negara", "city", "kota", "region", "wilayah"]):
        base += ["country", "city", "region"]
    if any(k in q for k in ["name", "nama"]):
        base += ["name"]
    if any(k in q for k in ["first name", "nama depan"]):
        base += ["first name", "contact first name"]
    if any(k in q for k in ["last name", "nama belakang", "surname"]):
        base += ["last name", "contact last name"]

    tokens = list(dict.fromkeys(base))[:max_tokens]

    if len(q.split()) <= 2:
        for t in ["amount", "price", "revenue", "month", "year"]:
            if t not in tokens and len(tokens) < max_tokens:
                tokens.append(t)

    probe_tail = " ".join(tokens).strip()
    return f"{question} {probe_tail}".strip()

# =========================
# === INGEST (PURE CHROMA)
# =========================
def _ingest_one_json(col, meta: dict, source_name: str, debug_id: str):
    """
    Upsert satu item ke koleksi Chroma.
    Return: (added, updated, skipped)
    """
    # ambil id
    doc_id = meta.get("field_urn") or meta.get("urn")
    if not doc_id:
        table = (meta.get("table") or meta.get("tableName") or "").strip()
        column = (meta.get("column") or meta.get("columnName") or "").strip()
        if not table and not column:
            return 0, 0, 1  # skip kalau minim info
        doc_id = f"{source_name}:{table}::{column}".lower()

    # normalisasi aliases & roles
    aliases = meta.get("tags_alias") or meta.get("aliases") or []
    roles   = meta.get("tags_role")  or meta.get("roles")   or []

    def _to_list(x):
        if isinstance(x, list):
            return [str(v) for v in x]
        if isinstance(x, str):
            try:
                j = json.loads(x)
                if isinstance(j, list):
                    return [str(v) for v in j]
                return [x]
            except Exception:
                return [x]
        return [str(x)]

    aliases = _to_list(aliases)
    roles   = _to_list(roles)

    table = (meta.get("table") or meta.get("tableName") or "").strip()
    column = (meta.get("column") or meta.get("columnName") or "").strip()
    desc = (meta.get("description") or "").strip()

    # rakit dokumen teks yang konsisten dengan yang sudah ada
    doc_text = (
        f"table: {table}\n"
        f"column: {column}\n"
        f"description: {desc}\n"
        f"aliases: {', '.join(aliases)}\n"
        f"roles: {', '.join(roles)}"
    )

    # simpan source & SANITASI metadata (PENTING)
    meta2 = dict(meta)
    meta2["_source"] = source_name
    meta2 = _sanitize_metadata(meta2)

    # cek eksistensi
    try:
        got = col.get(ids=[doc_id])
        exists = bool(got.get("ids"))
    except Exception:
        exists = False

    if exists:
        col.update(ids=[doc_id], documents=[doc_text], metadatas=[meta2])
        return 0, 1, 0
    else:
        col.add(ids=[doc_id], documents=[doc_text], metadatas=[meta2])
        return 1, 0, 0

def ingest_node(state: RAGState) -> RAGState:
    paths = state["jsonl_paths"]
    col = build_vector_store()

    added = updated = skipped = 0

    for path in paths:
        p = Path(path)
        if not p.exists():
            print(f"JSONL not found: {path}")
            continue

        for line_no, (raw, obj) in enumerate(read_jsonl(path), 1):
            if obj is None:
                try:
                    maybe = json.loads(raw)
                    if isinstance(maybe, list):
                        for i, item in enumerate(maybe):
                            if isinstance(item, dict):
                                _a, _u, _s = _ingest_one_json(
                                    col, item, p.name, f"{path}:{line_no}[{i}]"
                                )
                                added += _a
                                updated += _u
                                skipped += _s
                        continue
                except Exception:
                    pass
                print(f"[INGEST] skip {path}:{line_no} (not dict/list)")
                continue

            _a, _u, _s = _ingest_one_json(col, obj, p.name, f"{path}:{line_no}")
            added += _a
            updated += _u
            skipped += _s

    state["added"] = added
    state["updated"] = updated
    state["skipped"] = skipped
    return state

# =========================
# === COLUMN PRIOR BOOST ==
# =========================
def apply_column_priority_boost(
    q_l: str,
    table: str,
    col_name: str,
    roles: List[str],
    aliases: List[str],
) -> float:
    bonus = 0.0
    if any(k in q_l for k in ["name", "nama"]):
        if any(r == "name" or "name" in r for r in roles) or "name" in col_name:
            bonus += NAME_QUERY_BONUS
    if table.lower() == "customers":
        mentions_customer = any(k in q_l for k in ["customer", "customers", "pelanggan"])
        asks_id_like = any(k in q_l for k in [" id", "id ", "number", "nomor", "no.", "no ", "kode", "code", "customer number"])
        if mentions_customer and not asks_id_like:
            if "customername" in col_name or "customer_name" in col_name:
                bonus += 0.6
            if "customernumber" in col_name or col_name.endswith("number"):
                bonus -= 0.35
    if any(k in q_l for k in [" id ", "id ", " id,", "identifier", "kode", "code"]):
        if (
            any(r == "id" or "id" in r for r in roles)
            or "number" in col_name
            or col_name.lower().endswith("id")
        ):
            bonus += ID_QUERY_BONUS

    col_lower = col_name.lower()
    role_set = set(roles)
    alias_set = set(aliases)
    for rule in COLUMN_PRIORITY_CONFIG:
        tables = [t.lower() for t in rule.get("tables", []) or []]
        if tables and table.lower() not in tables:
            continue

        role_any = rule.get("role_any") or []
        if role_any and not any(r in role_set for r in role_any):
            continue

        alias_any = rule.get("alias_any") or []
        if alias_any and not any(a in alias_set for a in alias_any):
            continue

        column_contains = rule.get("column_contains") or []
        if column_contains and not any(sub in col_lower for sub in column_contains):
            continue

        qt_any = rule.get("query_terms_any") or []
        if qt_any and not any(t in q_l for t in qt_any):
            continue

        qt_also = rule.get("query_terms_also_any") or []
        if qt_also and not any(t in q_l for t in qt_also):
            continue

        bonus += float(rule.get("boost", 0.0))

    return bonus

# ============================================================
# === NEW: simple query classifier + table scorer ============
# ============================================================
def classify_query(q: str) -> str:
    """return: 'entity_list', 'entity_with_metric', or 'other'"""
    ql = q.lower()
    if any(kw in ql for kw in ["total", "count", "jumlah", "how many", "per ", "group by", "avg", "average", "sum"]):
        return "entity_with_metric"
    if any(kw in ql for kw in ["list", "show", "tampilkan", "daftar", "all"]):
        return "entity_list"
    toks = ql.split()
    if len(toks) <= 4 and any(t in ql for t in ["customer", "pelanggan", "product", "order"]):
        return "entity_list"
    return "other"

def score_tables_from_reranked(reranked: List[Dict[str, Any]], query_tokens: List[str]) -> Dict[str, float]:
    scores: Dict[str, float] = defaultdict(float)
    qt = [t.lower() for t in query_tokens]
    for it in reranked:
        md = it.get("meta") or {}
        tbl = (md.get("table") or md.get("tableName") or "").strip()
        if not tbl:
            continue
        desc = (md.get("description") or "").lower()
        base = 0.0
        if it.get("sparse"):
            base += 0.5
        if tbl.lower() in " ".join(qt):
            base += 1.0
        for t in qt:
            if t and t in desc:
                base += 0.3
        scores[tbl] += base
    return scores

# =========================
# === RETRIEVE (PURE CHROMA)
# =========================
def retrieve_node(state: RAGState) -> RAGState:
    col = build_vector_store()
    q = (state["question"] or "").strip()
    q_l = q.lower()

    qtype = classify_query(q)

    wants_customer_no_id = (
        any(w in q_l for w in ["customer", "customers", "pelanggan"])
        and not re.search(r"\b(id|number|nomor|no|code|kode)\b", q_l)
        and "customer number" not in q_l
    )

    jsonl_paths = state.get("jsonl_paths", [])
    sparse_field_urns = jsonl_sparse_search(q, jsonl_paths, max_hits=SPARSE_MAX_HITS)

    sparse_has_customername = any(
        u.lower().endswith("::customername") and ".customers," in u.lower()
        for u in sparse_field_urns
    )

    base_k = max(12, TOP_K)
    dense_k = base_k
    if wants_customer_no_id and not sparse_has_customername:
        dense_k = max(28, base_k)

    query_text = build_dynamic_probe(q)
    res = col.query(
        query_texts=[query_text],
        n_results=dense_k,
        include=["documents", "metadatas", "distances"],
    )

    dense_docs   = (res.get("documents")  or [[]])[0]
    dense_metas  = (res.get("metadatas")  or [[]])[0]
    dense_dists  = (res.get("distances")  or [[]])[0]
    dense_ids    = (res.get("ids")        or [[]])[0]

    n = min(
        len(dense_docs),
        len(dense_metas),
        len(dense_dists) if dense_dists else len(dense_docs),
        len(dense_ids) if dense_ids else len(dense_docs),
    )
    dense_docs  = dense_docs[:n]
    dense_metas = dense_metas[:n]
    dense_dists = (dense_dists or [0.0] * n)[:n]
    dense_ids   = (dense_ids or [""] * n)[:n]

    dense_has_customername = any(
        ((md.get("table") or md.get("tableName") or "").lower() == "customers")
        and ((md.get("column") or md.get("columnName") or "").lower() in ("customername", "customer_name"))
        for md in dense_metas
    )

    scored = []
    for rank, (doc_text, md, dist, _id) in enumerate(zip(dense_docs, dense_metas, dense_dists, dense_ids)):
        base = -float(dist) if dist is not None else 0.0
        sparse_hit = _doc_matches_sparse(md, sparse_field_urns)

        table_name = (md.get("table") or md.get("tableName") or "").strip()
        col_name   = (md.get("column") or md.get("columnName") or "").strip().lower()

        # parse roles
        roles_raw = md.get("tags_role") or "[]"
        roles: List[str] = []
        if isinstance(roles_raw, str):
            try:
                parsed = json.loads(roles_raw)
                if isinstance(parsed, list):
                    roles = [str(r).lower() for r in parsed]
                else:
                    roles = [str(roles_raw).lower()]
            except Exception:
                roles = [str(roles_raw).lower()]
        elif isinstance(roles_raw, list):
            roles = [str(r).lower() for r in roles_raw]

        # parse aliases
        aliases_raw = md.get("tags_alias") or "[]"
        aliases: List[str] = []
        if isinstance(aliases_raw, str):
            try:
                parsed_a = json.loads(aliases_raw)
                if isinstance(parsed_a, list):
                    aliases = [str(a).lower() for a in parsed_a]
                else:
                    aliases = [str(aliases_raw).lower()]
            except Exception:
                aliases = [str(aliases_raw).lower()]
        elif isinstance(aliases_raw, list):
            aliases = [str(a).lower() for a in aliases_raw]

        boosted = base + (SPARSE_BONUS if sparse_hit else 0.0)
        boosted += apply_column_priority_boost(
            q_l=q_l,
            table=table_name,
            col_name=col_name,
            roles=roles,
            aliases=aliases,
        )

        scored.append({
            "rank": rank,
            "id": _id,
            "doc": doc_text,
            "meta": md or {},
            "dist": float(dist) if dist is not None else 0.0,
            "base": base,
            "sparse": sparse_hit,
            "boosted": boosted,
        })

    # auto-inject sangat terbatas
    if wants_customer_no_id and sparse_has_customername and not dense_has_customername:
        sparse_customername_urn = None
        for u in sparse_field_urns:
            if u.lower().endswith("::customername"):
                sparse_customername_urn = u
                break
        if sparse_customername_urn:
            present_ids = {item["id"] for item in scored if item.get("id")}
            if sparse_customername_urn not in present_ids:
                got = col.get(ids=[sparse_customername_urn])
                if got and got.get("ids"):
                    md  = (got.get("metadatas") or [{}])[0]
                    doc = (got.get("documents") or [""])[0]

                    roles_raw = md.get("tags_role") or "[]"
                    roles: List[str] = []
                    if isinstance(roles_raw, str):
                        try:
                            parsed = json.loads(roles_raw)
                            if isinstance(parsed, list):
                                roles = [str(r).lower() for r in parsed]
                            else:
                                roles = [str(roles_raw).lower()]
                        except Exception:
                            roles = [str(roles_raw).lower()]
                    elif isinstance(roles_raw, list):
                        roles = [str(r).lower() for r in roles_raw]

                    aliases_raw = md.get("tags_alias") or "[]"
                    aliases: List[str] = []
                    if isinstance(aliases_raw, str):
                        try:
                            parsed_a = json.loads(aliases_raw)
                            if isinstance(parsed_a, list):
                                aliases = [str(a).lower() for a in parsed_a]
                            else:
                                aliases = [str(aliases_raw).lower()]
                        except Exception:
                            aliases = [str(aliases_raw).lower()]
                    elif isinstance(aliases_raw, list):
                        aliases = [str(a).lower() for a in aliases_raw]

                    table_name = (md.get("table") or md.get("tableName") or "").strip()
                    col_name   = (md.get("column") or md.get("columnName") or "").strip().lower()

                    max_boost = max((x["boosted"] for x in scored), default=0.0)
                    boosted = max_boost + 0.05
                    boosted += apply_column_priority_boost(
                        q_l=q_l,
                        table=table_name,
                        col_name=col_name,
                        roles=roles,
                        aliases=aliases,
                    )

                    scored.append({
                        "rank": len(scored),
                        "id": sparse_customername_urn,
                        "doc": doc,
                        "meta": md or {},
                        "dist": 0.0,
                        "base": 0.0,
                        "sparse": True,
                        "boosted": boosted,
                    })

    reranked = sorted(scored, key=lambda x: x["boosted"], reverse=True)
    reranked = reranked[:MAX_CTX_DOCS]

    if RETRIEVE_DEBUG:
        print("\n===== RETRIEVE DEBUG =====")
        print(f"Question         : {q}")
        print(f"Query (dense)    : {query_text}")
        print(f"Dense docs (raw) : {len(scored)}")
        print(f"Sparse query (clean) : {build_sparse_query_from_question(q)!r}")
        print("After rerank (trimmed):")
        for i, item in enumerate(reranked):
            md   = item["meta"]
            tbl  = md.get("table") or md.get("tableName") or ""
            col  = md.get("column") or md.get("columnName") or ""
            print(
                f"  • final_rank={i + 1}  boosted={item['boosted']:.4f}  "
                f"match_sparse={item['sparse']}  -> table={tbl}  column={col}"
            )
        print("===== END RETRIEVE DEBUG =====\n")

    # build allowed + alias_map
    allowed   = defaultdict(list)
    alias_map = defaultdict(dict)
    for item in reranked:
        md = item["meta"] or {}
        t  = (md.get("table") or md.get("tableName") or "").strip()
        c  = (md.get("column") or md.get("columnName") or "").strip()
        if t and c:
            allowed[t].append(c)
            aliases2 = md.get("aliases") or md.get("tags_alias") or md.get("synonyms") or []
            if isinstance(aliases2, str):
                try:
                    maybe = json.loads(aliases2)
                    if isinstance(maybe, list):
                        aliases2 = maybe
                    else:
                        aliases2 = [aliases2]
                except Exception:
                    aliases2 = [aliases2]
            for al in aliases2:
                al = str(al).strip()
                if al:
                    alias_map[t][al.lower()] = c

    token_for_score = _tokenize_bm25_query(q)
    table_scores = score_tables_from_reranked(reranked, token_for_score)

    if classify_query(q) == "entity_list" and wants_customer_no_id:
        filtered_allowed = {t: cols for t, cols in allowed.items() if t.lower() == "customers"}
    else:
        if table_scores:
            sorted_tbls = sorted(table_scores.items(), key=lambda x: x[1], reverse=True)
            top_tables = {name for name, score in sorted_tbls[:2] if score > 0.0}
            if top_tables:
                filtered_allowed = {t: cols for t, cols in allowed.items() if t in top_tables}
            else:
                filtered_allowed = dict(allowed)
        else:
            filtered_allowed = dict(allowed)

    if wants_customer_no_id:
        for t_key in list(filtered_allowed.keys()):
            if t_key.lower() == "customers":
                if "customerName" not in filtered_allowed[t_key]:
                    filtered_allowed[t_key].append("customerName")

    allowed_lines = []
    for t, cols in filtered_allowed.items():
        uniq = sorted(set(cols))
        allowed_lines.append(f"- {t}: {', '.join(uniq)}")
    allowed_block = "### ALLOWED IDENTIFIERS\n" + ("\n".join(allowed_lines) if allowed_lines else "- (none)")

    cards: List[str] = []
    present = set()
    for it in reranked:
        md = it["meta"] or {}
        table = (md.get("table") or md.get("tableName") or "").strip()
        column = (md.get("column") or md.get("columnName") or "").strip()
        if table not in filtered_allowed:
            continue
        desc = (md.get("description") or "").strip()
        if len(desc) > 160:
            desc = desc[:160] + "..."
        cards.append(f"- table: {table} | column: {column} | desc: {desc}")
        present.add((table.lower(), column.lower()))

    injected_cards = []
    for tbl, cols in filtered_allowed.items():
        for col in cols:
            key = (tbl.lower(), col.lower())
            if key not in present:
                short_desc = ""
                if tbl.lower() == "customers" and col.lower() == "customername":
                    short_desc = "customer display name / company name"
                injected_cards.append(f"- table: {tbl} | column: {col} | desc: {short_desc}")
                present.add(key)

    cards = injected_cards + cards
    cards_text = "\n".join(cards)

    context = (allowed_block + "\n\n" + cards_text).strip()
    context += f"\n\n### _INTERNAL_HINT\n{json.dumps({'alias_map': alias_map}, ensure_ascii=False)}"

    state["context"] = context
    return state

# =========================
# === SQL POSTPROCESS =====
# =========================
_JOIN_KW = r"(?:inner\s+join|left\s+join|right\s+join|full\s+join|join)"

def _has_join(sql: str) -> bool:
    return re.search(rf"\b{_JOIN_KW}\b", sql, flags=re.I) is not None

def _should_use_backtick(identifier: str) -> bool:
    if not identifier: return False
    clean = identifier.strip('`"')
    if not clean: return False
    first_char = clean[0]
    if first_char.isupper() or first_char.isdigit():
        return True
    if not re.match(r'^[a-z_][a-z0-9_]*$', clean, re.I):
        return True
    return False

def _smart_backtick(identifier: str) -> str:
    clean = identifier.strip('`"')
    return f"`{clean}`" if _should_use_backtick(clean) else clean

def _split_qualified(name: str) -> List[str]:
    return [p.strip('`" ') for p in name.split(".") if p.strip("\"` ")]

def _fix_bad_backticks(sql: str) -> str:
    def repl_fqn(m: re.Match) -> str:
        inside = m.group(1)
        if "." not in inside:
            return _smart_backtick(inside)
        parts = _split_qualified(inside)
        if len(parts) == 0:
            return inside
        return ".".join(_smart_backtick(p) for p in parts)
    sql = re.sub(r"`([^`]+?\.[^`]+?)`", repl_fqn, sql)
    sql = re.sub(r"(\b[A-Za-z_][\w]*)\.`([^`]+?)`",
                 lambda m: f"{_smart_backtick(m.group(1))}."
                           + ".".join(_smart_backtick(p) for p in _split_qualified(m.group(2))), sql)
    sql = re.sub(r"`([^`]+?)`\.(\b[A-Za-z_][\w]*)",
                 lambda m: ".".join(_smart_backtick(p) for p in _split_qualified(m.group(1)))
                           + f".{_smart_backtick(m.group(2))}", sql)
    sql = re.sub(r'`([^`]+?)`(?!\s*\.)', lambda m: _smart_backtick(m.group(1)), sql, flags=re.I)
    return sql

def _explicit_limit_from_question(question: str) -> Optional[int]:
    q = (question or "").lower()
    m = re.search(r"\blimit\s+(\d{1,4})\b", q)
    if m: return max(1, min(int(m.group(1)), 1_000_000))
    m = re.search(r"\btop\s+(\d{1,4})\b", q)
    if m: return max(1, min(int(m.group(1)), 1_000_000))
    m = re.search(r"\b(show|first|tampilkan|ambil)\s+(\d{1,4})\b", q)
    if m: return max(1, min(int(m.group(2)), 1_000_000))
    return None

def _maybe_add_limit(question: str, sql: str) -> str:
    if re.search(r"\blimit\s+\d+\b", sql, flags=re.I):
        return sql
    n = _explicit_limit_from_question(question)
    if n is None:
        return sql
    s = sql.rstrip().rstrip(";")
    return s + f" LIMIT {n};"

def _wants_distinct_from_question(question: str) -> bool:
    q = (question or "").lower()
    hard_signals = [
        "distinct",
        "no duplicate",
        "without duplicates",
        "tanpa duplikat",
        "tanpa duplicate",
        "hanya yang unik",
        "dedup",
        "de-dup",
    ]
    if any(s in q for s in hard_signals):
        return True
    if "unique" in q and ("rows" in q or "records" in q or "result" in q):
        return True
    return False

def _has_distinct(sql: str) -> bool:
    return bool(re.search(r"^\s*select\s+distinct\b", sql, flags=re.I))

def _add_distinct(sql: str) -> str:
    if _has_distinct(sql):
        return sql
    return re.sub(r"^\s*select\s+", "SELECT DISTINCT ", sql, flags=re.I)

def _remove_distinct(sql: str) -> str:
    return re.sub(r"^\s*select\s+distinct\s+", "SELECT ", sql, flags=re.I)

def _ensure_reasonable_distinct(sql: str, question: str) -> str:
    if not sql:
        return sql
    if _wants_distinct_from_question(question):
        return _add_distinct(sql)
    return _remove_distinct(sql)

def _fix_missing_space_before_clauses(sql: str) -> str:
    sql = re.sub(r"\s+(?=(ORDER\s+BY|GROUP\s+BY|HAVING|LIMIT|WHERE)\b)", " ", sql, flags=re.I)
    sql = re.sub(
        r"([)\],;`])(?=(ORDER\s+BY|GROUP\s+BY|HAVING|LIMIT|WHERE)\b)",
        r"\1 ",
        sql,
        flags=re.I,
    )
    sql = re.sub(r"([A-Za-z0-9_\.])(?=(LIMIT|WHERE|HAVING)\b)", r"\1 ", sql)
    sql = re.sub(r"([A-Za-z0-9_\.])(?=ORDER\s+BY\b)", r"\1 ", sql)
    sql = re.sub(r"([A-Za-z0-9_\.])(?=GROUP\s+BY\b)", r"\1 ", sql)
    sql = re.sub(r"\s+(ORDER\s+BY|GROUP\s+BY|HAVING|LIMIT|WHERE)\b", r" \1", sql, flags=re.I)
    return sql

def _collapse_spaces_outside_quotes(s: str) -> str:
    parts = re.split(r"('(?:''|[^'])*')", s)
    for i in range(len(parts)):
        if i % 2 == 0:
            parts[i] = re.sub(r"[ \t]{2,}", " ", parts[i])
    return "".join(parts)

def _extract_order_by_clause(sql: str) -> Optional[re.Match]:
    return re.search(r"\border\s+by\s+(.*?)(?=(\blimit\b|;|$))", sql, flags=re.I | re.S)

def _remove_order_by_clause(sql: str) -> str:
    return re.sub(r"\s+order\s+by\s+.*?(?=(\blimit\b|;|$))", "", sql, flags=re.I | re.S)

def _question_requests_ordering(question: str) -> bool:
    q = (question or "").lower()
    keywords = [
        "order by", "sort", "sorted", "ascending", "descending", "asc", "desc",
        "alphabetical", "alphabetically", "latest", "newest", "oldest",
        "highest", "lowest", "top", "bottom", "most expensive", "cheapest",
        "urut", "urutkan", "diurutkan", "menaik", "menurun",
        "terbesar", "terkecil", "tertinggi", "terendah", "terbaru", "terlama",
        "termahal", "termurah",
    ]
    return any(k in q for k in keywords)

def _question_prefers_customer_name(question: str) -> bool:
    q = (question or "").lower()
    q = re.sub(r"\b(total\s+)?number\s+of\b", "", q)
    mention_customer = any(w in q for w in ["customer", "customers", "pelanggan"])
    mention_id_like = re.search(r"\b(id|number|nomor|no|code|kode)\b", q) or "customer number" in q
    return mention_customer and not mention_id_like

def _prefer_name_over_number_for_customers(sql: str, question: str) -> str:
    if not _question_prefers_customer_name(question):
        return sql

    s_new = re.sub(
        r"SELECT\s+c\.customerNumber(\s+AS\s+customerNumber)?",
        "SELECT c.customerName AS customerName",
        sql,
        flags=re.I,
    )

    s_new = re.sub(
        r"SELECT\s+c\.customerNumber\s*,\s*COUNT\(",
        "SELECT c.customerName, COUNT(",
        s_new,
        flags=re.I,
    )

    s_new = re.sub(
        r"GROUP BY\s+c\.customerNumber\b",
        "GROUP BY c.customerName",
        s_new,
        flags=re.I,
    )

    return s_new

def _strip_unneeded_count_distinct(sql: str) -> str:
    low = sql.lower()
    if "from customers" in low and "join orders" in low:
        sql = re.sub(
            r"COUNT\s*\(\s*DISTINCT\s+o\.orderNumber\s*\)",
            "COUNT(o.orderNumber)",
            sql,
            flags=re.IGNORECASE,
        )
    return sql

def _postprocess_sql(sql: str, question: str, context: str = "") -> str:
    if not sql:
        return sql
    s = sql.strip()

    def _norm_kw(text: str) -> str:
        CHUNKS = re.compile(
            r"(`[^`]*`|\"([^\"\\]|\\.)*\"|'([^'\\]|\\.)*')",
            re.DOTALL,
        )
        KEYWORDS = [
            "select","from","where","group by","order by","having","limit",
            "inner join","left join","right join","full join","join","on",
        ]

        def normalize_code_segment(seg: str) -> str:
            for kw in KEYWORDS:
                pattern = rf"(?<![A-Za-z0-9_]){kw}(?![A-Za-z0-9_])"
                seg = re.sub(pattern, lambda m: m.group(0).upper(), seg, flags=re.I)
            seg = re.sub(r"[ \t]+", " ", seg)
            return seg

        out, i = [], 0
        for m in CHUNKS.finditer(text):
            out.append(normalize_code_segment(text[i:m.start()]))
            out.append(m.group(0))
            i = m.end()
        out.append(normalize_code_segment(text[i:]))

        return "".join(out)

    s = _norm_kw(s)
    if not s.endswith(";"):
        s += ";"

    s = _fix_bad_backticks(s)
    s = _ensure_reasonable_distinct(s, question)
    s = _prefer_name_over_number_for_customers(s, question)
    ob_match = _extract_order_by_clause(s)
    if ob_match and not _question_requests_ordering(question):
        s = _remove_order_by_clause(s)

    s = _maybe_add_limit(question, s)
    s = _fix_missing_space_before_clauses(s)
    s = _collapse_spaces_outside_quotes(s)
    s = _strip_unneeded_count_distinct(s)
    return s

# ---------- Whitelist parsing + alias repair ----------
_ALNUM = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")

def _extract_internal_hint(ctx: str) -> Dict[str, Dict[str, str]]:
    m = re.search(r"### _INTERNAL_HINT\s+({.*})\s*$", ctx, flags=re.S)
    if not m:
        return {}
    try:
        obj = json.loads(m.group(1))
        return obj.get("alias_map") or {}
    except Exception:
        return {}

def _parse_allowed_identifiers(ctx: str) -> Dict[str, set]:
    tables: Dict[str, set] = {}
    m = re.search(r"### ALLOWED IDENTIFIERS(.*?)(?:\n### |\Z)", ctx, flags=re.S)
    if not m:
        return tables
    block = m.group(1)
    for line in block.splitlines():
        line = line.strip()
        if not line.startswith("- "):
            continue
        parts = line[2:].split(":", 1)
        if len(parts) != 2:
            continue
        table = parts[0].strip().strip("`")
        cols = [c.strip().strip("`") for c in parts[1].split(",") if c.strip()]
        if table:
            tables.setdefault(table, set()).update(cols)
    return tables

def _extract_tables_from_sql(sql: str) -> set:
    tables = set()
    for m in re.finditer(r"\bfrom\s+([A-Za-z_][A-Za-z0-9_]*)", sql, flags=re.I):
        tables.add(m.group(1))
    for m in re.finditer(r"\bjoin\s+([A-Za-z_][A-Za-z0-9_]*)", sql, flags=re.I):
        tables.add(m.group(1))
    return tables

def _identify_used_identifiers(sql: str) -> Dict[Optional[str], set]:
    qualified = re.findall(
        r"`?([A-Za-z_][A-Za-z0-9_]*)`?\s*\.\s*`?([A-Za-z_][A-Za-z0-9_]*)`?",
        sql,
    )
    out: Dict[Optional[str], set] = {}
    for t, c in qualified:
        out.setdefault(t, set()).add(c)

    table_names = _extract_tables_from_sql(sql)
    for t in table_names:
        out.setdefault(t, set())

    keywords = {
        "select","from","where","group","by","order","limit","join","left","right",
        "inner","on","and","or","not","as","distinct","count","sum","avg","min","max",
        "having","asc","desc","like","in","between","is","null","case","when","then",
        "else","end"
    }

    for tok in re.findall(r"\b([A-Za-z_][A-Za-z0-9_]*)\b", sql):
        if tok.lower() in keywords:
            continue
        if tok in table_names:
            continue
        out.setdefault(None, set()).add(tok)

    return out

def _validate_identifiers(sql: str, ctx: str) -> List[str]:
    allowed = _parse_allowed_identifiers(ctx)
    used = _identify_used_identifiers(sql)
    errs: List[str] = []

    explicit_tables = {t for t in used.keys() if t is not None}

    single_table: Optional[str] = None
    if len(explicit_tables) == 1:
        single_table = next(iter(explicit_tables))
    elif len(explicit_tables) == 0 and len(allowed) == 1:
        single_table = next(iter(allowed.keys()))

    for tbl, cols in used.items():
        if tbl is None:
            continue
        if tbl not in allowed:
            errs.append(f"table `{tbl}` not in ALLOWED")
            continue
        bad = [c for c in cols if c not in allowed[tbl]]
        if bad:
            errs.append(f"{tbl} -> invalid columns: {', '.join(bad)}")

    if single_table:
        allowed_cols = allowed.get(single_table, set())
        for col in used.get(None, set()):
            if col not in allowed_cols:
                errs.append(f"{single_table} -> invalid column (unqualified): {col}")

    return errs

def _repair_with_aliases(sql: str, ctx: str) -> str:
    """
    Perbaiki nama kolom berdasar alias_map dan fallback classicmodels,
    TAPI jangan menyentuh teks dalam kutip ('...', "...", `...`)
    supaya alias/label buatan LLM tetap utuh.
    """
    alias_map_per_table = _extract_internal_hint(ctx)

    # global map (lower → canonical)
    global_map: Dict[str, Optional[str]] = {}
    for _t, amap in (alias_map_per_table or {}).items():
        for al, canon in (amap or {}).items():
            al = str(al).lower()
            if al not in global_map:
                global_map[al] = canon
            elif global_map[al] != canon:
                global_map[al] = None

    # fallback untuk classicmodels
    classic_fallback = {
        "products": {
            "name": "productName",
            "productname": "productName",
            "listprice": "MSRP",
        },
        "orderdetails": {
            "listprice": "priceEach",
            "price": "priceEach",
        },
        "customers": {
            "name": "customerName",
            "customername": "customerName",
            "id": "customerNumber",
            "customerid": "customerNumber",
            "customer_id": "customerNumber",
            "id_customer": "customerNumber",
            "kodecustomer": "customerNumber",
            "kode_pelanggan": "customerNumber",
        },
    }
    for _t, amap in classic_fallback.items():
        for al, canon in amap.items():
            al = al.lower()
            if al not in global_map:
                global_map[al] = canon
            elif global_map[al] is None:
                pass

    keywords = {
        "select","from","where","group","by","order","limit","join","left","right","inner",
        "on","and","or","not","as","distinct","count","sum","avg","min","max","having",
        "asc","desc","like","in","between","is","null","case","when","then","else","end"
    }
    token_pat = re.compile(r"\b([A-Za-z_][A-Za-z0-9_]*)\b")

    def replace_in_code_segment(seg: str) -> str:
        def smart_repl(m):
            w = m.group(0)
            lw = w.lower()
            if lw in keywords:
                return w
            canon = global_map.get(lw)
            if not canon:
                return w
            if not _ALNUM.match(canon):
                return w
            return canon
        return token_pat.sub(smart_repl, seg)

    parts = re.split(r"('(?:''|[^'])*'|\"(?:\"\"|[^\"])*\"|`[^`]*`)", sql)
    out_parts = []
    for idx, part in enumerate(parts):
        if idx % 2 == 1:
            out_parts.append(part)  # quoted → jangan disentuh
        else:
            out_parts.append(replace_in_code_segment(part))  # code → boleh direpair
    return "".join(out_parts)

# ---------- Prompt ----------
def _make_generation_prompt(ctx: str, question: str) -> str:
    return f"""
You are a senior SQL assistant for MySQL 8.0.

Return a runnable query using ONLY tables/columns listed in the ALLOWED IDENTIFIERS section of Context.
If a needed column name appears as an alias in Context, use its canonical column name.
If the query is about customers and the user does NOT explicitly ask for an id/number/code, prefer customerName over customerNumber AND also prefer customerName over contactFirstName/contactLastName when both are available in ALLOWED IDENTIFIERS.

########################
# STRICT OUTPUT FORMAT #
########################
{{ "sql": "<final MySQL query>", "notes": "<brief rationale; for EACH JOIN, name the exact selected/grouped/filtered column it contributes>" }}
# No extra keys. No Markdown. End SQL with a semicolon.
# If the final SQL has NO JOINs, set "notes" exactly to: "No joins used."

##################
# SQL CONSTRAINT #
##################
- MySQL 8.0; backticks only when needed; ANSI joins.

################
# CONTEXT
################
{ctx}

Question:
{question}
""".strip()

def _sanitize_notes(sql: str, notes: str) -> str:
    if not notes:
        return notes
    if not _has_join(sql or ""):
        return "No joins used."
    return notes.strip()

# ---------- extra helpers for generation robustness ----------
def _try_llm_json_three_times(llm: ChatOllama, base_prompt: str) -> Optional[Dict[str, Any]]:
    prompts = [
        base_prompt,
        base_prompt + "\n\nReturn ONLY the JSON object. Do not add explanation.",
        base_prompt + "\n\nReturn ONLY JSON with keys \"sql\" and \"notes\". No extra text. Example: {\"sql\":\"SELECT 1;\",\"notes\":\"No joins used.\"}",
    ]
    for p in prompts:
        resp = llm.invoke(p)
        content = getattr(resp, "content", None) or str(resp)
        try:
            obj = json.loads(content)
            if isinstance(obj, dict) and ("sql" in obj or "notes" in obj):
                return obj
        except Exception:
            m = re.search(r"\{.*\}", content, flags=re.S)
            if m:
                try:
                    obj = json.loads(m.group(0))
                    if isinstance(obj, dict) and ("sql" in obj or "notes" in obj):
                        return obj
                except Exception:
                    pass
    return None

def _heuristic_sql_from_question(question: str, ctx: str) -> Optional[str]:
    q = (question or "").lower()
    if "customer" in q and "order" in q and ("total" in q or "number of" in q):
        return (
            "SELECT c.customerName, COUNT(o.orderNumber) AS totalOrders\n"
            "FROM customers c\n"
            "LEFT JOIN orders o ON o.customerNumber = c.customerNumber\n"
            "GROUP BY c.customerName\n"
            "ORDER BY totalOrders DESC;"
        )
    return None

# ---------- Generation (with validation & regeneration) ----------
def generate_node(state: RAGState) -> RAGState:
    llm = ChatOllama(
        model=CHAT_MODEL,
        base_url=OLLAMA_BASE_URL,
        temperature=0.0,
        seed=42
    )
    ctx = state.get("context", "")
    question = state.get("question", "")
    base_prompt = _make_generation_prompt(ctx, question)

    answer_out = ""
    draft_sql = ""
    final_sql = ""
    notes = ""

    obj = _try_llm_json_three_times(llm, base_prompt)

    if obj is not None:
        draft_sql = (obj.get("sql") or "").strip()
        notes = (obj.get("notes") or "").strip()
    else:
        heuristic = _heuristic_sql_from_question(question, ctx)
        if heuristic:
            draft_sql = heuristic
            notes = "Heuristic fallback for customers + orders."
        else:
            draft_sql = "SELECT 1;"
            notes = "Heuristic trivial fallback. No joins used."

    if draft_sql:
        _dbg_print_two("RAG SQL Code (before postprocess) :", draft_sql)

    # 1) postprocess awal
    final_sql = _postprocess_sql(draft_sql, question, ctx) if draft_sql else draft_sql

    # 2) repair alias tapi aman terhadap quoted alias
    if final_sql:
        final_sql = _repair_with_aliases(final_sql, ctx)

    # 3) postprocess lagi buat rapihin spacing / order by
    if final_sql:
        final_sql = _postprocess_sql(final_sql, question, ctx)

    # validasi allowed identifiers
    violations = _validate_identifiers(final_sql or "", ctx)
    if violations:
        force_msg = (
            "The previous SQL used identifiers not present in ALLOWED IDENTIFIERS:\n- "
            + "\n- ".join(violations)
            + "\nRewrite the SQL using ONLY identifiers listed in ALLOWED IDENTIFIERS. "
              "If a synonym appears, choose the canonical column."
        )
        strict_prompt = _make_generation_prompt(ctx + "\n\n" + force_msg, question)
        obj2 = _try_llm_json_three_times(llm, strict_prompt)
        if obj2 is not None:
            draft2 = (obj2.get("sql") or "").strip()
            notes2 = (obj2.get("notes") or "").strip()
        else:
            draft2 = ""
            notes2 = ""
        sql2 = _postprocess_sql(draft2 or "", question, ctx) if draft2 else ""
        if sql2:
            sql2 = _repair_with_aliases(sql2, ctx)
            sql2 = _postprocess_sql(sql2, question, ctx)
        v1 = len(violations)
        v2 = len(_validate_identifiers(sql2 or "", ctx))
        if v2 <= v1 and sql2:
            final_sql, notes = sql2, (notes2 or notes)

    if final_sql:
        _dbg_print_two("Sanitize SQL Code :", final_sql)

    notes = _sanitize_notes(final_sql or "", notes or "")

    if final_sql:
        answer_out = final_sql + ("\n\n-- notes: " + notes if notes else "")
    else:
        answer_out = draft_sql or "SELECT 1;"

    state["answer"] = answer_out
    state["draft_sql"] = draft_sql
    state["final_sql"] = final_sql
    return state

# =========================
# === APP GRAPH ===========
# =========================
def build_app():
    g = StateGraph(RAGState)
    g.add_node("ingest", ingest_node)
    g.add_node("retrieve", retrieve_node)
    g.add_node("generate", generate_node)
    g.set_entry_point("ingest")
    g.add_edge("ingest", "retrieve")
    g.add_edge("retrieve", "generate")
    g.add_edge("generate", END)
    return g.compile()

# =========================
# === MCP UTIL ============
# =========================
async def _mcp_call(client: Client, tool: str, payload: dict | None = None, timeout: int = MCP_TIMEOUT):
    coro = client.call_tool(tool, payload or {})
    return await asyncio.wait_for(coro, timeout=timeout)

def _extract_payload(result) -> Any:
    if hasattr(result, 'data') and result.data is not None:
        return result.data
    if hasattr(result, "structuredContent") and result.structuredContent is not None:
        return result.structuredContent
    if hasattr(result, "content") and result.content:
        return result.content[0].text if isinstance(result.content, list) else str(result.content)
    return result

def _ensure_json(d: Any) -> Optional[dict]:
    if isinstance(d, dict):
        return d
    if isinstance(d, str):
        try:
            return json.loads(d)
        except Exception:
            return None
    return None

def display_tables_list(data: dict):
    tables = data.get("tables", [])
    total = data.get("total_tables", len(tables))
    if not tables:
        print("No tables found.")
        return
    print("\nAvailable tables:")
    for i, t in enumerate(tables, start=1):
        print(f"  {i:>2}. {t}")
    print(f"\nTotal: {total} table(s)")

def display_table_from_json(data):
    preview_rows = data.get("preview_rows", [])
    preview_cols = data.get("preview_columns", [])
    total_rows = data.get("total_rows", len(preview_rows))
    total_cols = data.get("total_columns", len(preview_cols))

    if preview_rows and preview_cols:
        print("\nQuery Result (preview):")
        print(
            tabulate(
                preview_rows,
                headers=preview_cols,
                tablefmt="github",
                showindex=range(1, len(preview_rows) + 1),
            )
        )
        print(
            f"\nShowing preview: {len(preview_rows)} rows × {len(preview_cols)} columns "
            f"(full result: {total_rows} rows × {total_cols} columns)"
        )
    else:
        print("Query executed but returned no rows.")

# =========================
# === REPL RUNNER =========
# =========================
async def repl(paths: List[str]):
    app = build_app()

    transport = StreamableHttpTransport(MCP_BASE)
    async with Client(transport=transport) as mcp:
        first_q = input("Enter a test question (or press Enter to skip): ").strip()
        if not first_q:
            first_q = "Which columns describe customer identity?"

        t0_total = _now()  # <<< ADD: start total timing untuk first run

        init: RAGState = {
            "jsonl_paths": paths,
            "question": first_q,
            "added": 0,
            "updated": 0,
            "skipped": 0,
            "context": "",
            "answer": "",
            "draft_sql": "",
            "final_sql": "",
        }

        out = app.invoke(init)

        col = build_vector_store()
        total = col.count()
        print("\n=== RAG Ingest Report ===")
        print(f"Added (first run): {out.get('added', 0)}")
        print(f"Updated (upserted): {out.get('updated', 0)}")
        print(f"Skipped (unchanged): {out.get('skipped', 0)}")
        print(f"Total docs now in collection: {total}")
        print(f"Chroma path: {Path(CHROMA_DIR).resolve()} | Collection: {COLLECTION_NAME}")

        print("\n--- Generated Answer (RAG) ---\n")
        generated = out.get("answer", "(no answer)")
        print(generated)

        async def exec_via_mcp(sql_block: str, final_sql: Optional[str] = None):
            sql_only = sql_block.split("\n\n-- notes:")[0].strip()
            sql_clean = (final_sql or sql_only).rstrip(";").strip()

            try:
                lt = await _mcp_call(mcp, "list_tables", {})
                lt_payload = _ensure_json(_extract_payload(lt)) or {}
                if lt_payload.get("success"):
                    display_tables_list(lt_payload)
            except Exception as e:
                print(f"list_tables failed (skip): {e}")

            print(f"\nSQL (RAG -> MCP):\n{sql_clean};")

            try:
                res = await _mcp_call(mcp, "query_db", {"sql": sql_clean})
                payload = _ensure_json(_extract_payload(res))
                if isinstance(payload, dict) and payload.get("success"):
                    display_table_from_json(payload)
                    return payload
                else:
                    err = payload.get("error") if isinstance(payload, dict) else str(payload)
                    print(f"query_db failed: {err}")
                    return None
            except asyncio.TimeoutError:
                print("MCP query_db timeout")
                return None
            except Exception as e:
                print(f"MCP query_db error: {e}")
                return None

        await exec_via_mcp(generated, out.get("final_sql"))

        _log_total("Question → Result (first run)", t0_total)  # <<< ADD: stop & print total

        print("\nAsk me more questions! (type 'exit' to quit)")
        while True:
            q = input("\nQuestion> ").strip()
            if not q or q.lower() in {"exit", "quit", "q"}:
                print("Bye.")
                break

            t0_total = _now()  # <<< ADD: start total timing untuk pertanyaan ini

            state: RAGState = {
                "jsonl_paths": paths,
                "question": q,
                "added": 0,
                "updated": 0,
                "skipped": 0,
                "context": "",
                "answer": "",
                "draft_sql": "",
                "final_sql": "",
            }
            state = retrieve_node(state)
            state = generate_node(state)

            print("\n--- Generated Answer (RAG) ---\n")
            print(state.get("answer", "(no answer)"))

            await exec_via_mcp(state.get("answer", ""), state.get("final_sql"))

            _log_total("Question → Result", t0_total)  # <<< ADD: stop & print total

def main():
    paths = resolve_jsonl_paths(JSON_INPUT)
    if not paths:
        raise RuntimeError(
            "No JSONL inputs found.\n"
            "Set EXPORT_JSONL_OUTPUT to one of:\n"
            "  • a single file: ./exports/classicmodels.jsonl\n"
            "  • a directory:   ./datahub_metadata\n"
            "  • a glob:        ./datahub_metadata/*.jsonl\n"
        )

    try:
        asyncio.run(repl(paths))
    except RuntimeError as e:
        if "asyncio.run() cannot be called" in str(e):
            import nest_asyncio
            nest_asyncio.apply()
            loop = asyncio.get_event_loop()
            loop.run_until_complete(repl(paths))
        else:
            raise

if __name__ == "__main__":
    main()

Enter a test question (or press Enter to skip):  List all customer names (alphabetical).



===== RETRIEVE DEBUG =====
Question         : List all customer names (alphabetical).
Query (dense)    : List all customer names (alphabetical). customer customer name customerName name
Dense docs (raw) : 12
Sparse query (clean) : 'list all customer names alphabetical'
After rerank (trimmed):
  • final_rank=1  boosted=-422.3255  match_sparse=True  -> table=customers  column=customerName
  • final_rank=2  boosted=-440.8403  match_sparse=True  -> table=orders  column=customerNumber
  • final_rank=3  boosted=-461.3417  match_sparse=True  -> table=customers  column=contactFirstName
  • final_rank=4  boosted=-471.4603  match_sparse=True  -> table=customers  column=contactLastName
  • final_rank=5  boosted=-473.6266  match_sparse=False  -> table=customers  column=customerNumber
  • final_rank=6  boosted=-483.2089  match_sparse=False  -> table=payments  column=customerNumber
  • final_rank=7  boosted=-499.9252  match_sparse=True  -> table=products  column=productName
  • final_rank=8  booste


Question>  Show distinct customer countries.



===== RETRIEVE DEBUG =====
Question         : Show distinct customer countries.
Query (dense)    : Show distinct customer countries. id date status count qty quantity number customer customer name customerName
Dense docs (raw) : 12
Sparse query (clean) : 'show distinct customer countries'
After rerank (trimmed):
  • final_rank=1  boosted=-411.8951  match_sparse=True  -> table=customers  column=country
  • final_rank=2  boosted=-482.1641  match_sparse=True  -> table=customers  column=customerNumber
  • final_rank=3  boosted=-515.9041  match_sparse=False  -> table=customers  column=state
  • final_rank=4  boosted=-517.0155  match_sparse=True  -> table=orders  column=customerNumber
  • final_rank=5  boosted=-521.7782  match_sparse=True  -> table=payments  column=customerNumber
  • final_rank=6  boosted=-523.0452  match_sparse=True  -> table=customers  column=city
  • final_rank=7  boosted=-526.5430  match_sparse=True  -> table=customers  column=customerName
  • final_rank=8  boosted=-592


Question>  How many orders exist?



===== RETRIEVE DEBUG =====
Question         : How many orders exist?
Query (dense)    : How many orders exist? id date status count qty quantity number
Dense docs (raw) : 12
Sparse query (clean) : 'how many orders exist'
After rerank (trimmed):
  • final_rank=1  boosted=-513.7300  match_sparse=True  -> table=orderdetails  column=quantityOrdered
  • final_rank=2  boosted=-514.2535  match_sparse=False  -> table=products  column=quantityInStock
  • final_rank=3  boosted=-542.8782  match_sparse=True  -> table=orders  column=status
  • final_rank=4  boosted=-556.9615  match_sparse=True  -> table=orders  column=orderNumber
  • final_rank=5  boosted=-569.8291  match_sparse=True  -> table=orders  column=orderDate
  • final_rank=6  boosted=-575.0687  match_sparse=False  -> table=orders  column=shippedDate
  • final_rank=7  boosted=-592.8278  match_sparse=True  -> table=orderdetails  column=orderNumber
  • final_rank=8  boosted=-603.9204  match_sparse=False  -> table=orders  column=customerNumb

... (8 outputs left out)


Question>  List all customers and the total number of orders they have placed.



===== RETRIEVE DEBUG =====
Question         : List all customers and the total number of orders they have placed.
Query (dense)    : List all customers and the total number of orders they have placed. id date status sum total amount revenue sales price gross net customer customer name
Dense docs (raw) : 12
Sparse query (clean) : 'list all customers and the total number of orders they have placed'
After rerank (trimmed):
  • final_rank=1  boosted=-466.7240  match_sparse=True  -> table=orders  column=customerNumber
  • final_rank=2  boosted=-479.4768  match_sparse=True  -> table=customers  column=customerNumber
  • final_rank=3  boosted=-485.8887  match_sparse=True  -> table=payments  column=customerNumber
  • final_rank=4  boosted=-502.9742  match_sparse=True  -> table=customers  column=customerName
  • final_rank=5  boosted=-508.1539  match_sparse=True  -> table=orders  column=orderNumber
  • final_rank=6  boosted=-511.6207  match_sparse=True  -> table=orderdetails  column=orderNumber


Question>  exit


Bye.


## **CHECK COLLECTION**

In [ ]:
from pprint import pprint
import chromadb
import numpy as np  # ok if installed; fallback guards below handle absence

CHROMA_DIR = "./chroma_db"
COLLECTION_NAME = "classicmodels"

client = chromadb.PersistentClient(path=CHROMA_DIR)
col = client.get_collection(COLLECTION_NAME)

print("== BASIC INFO ==")
print("name :", col.name)
print("count:", col.count())

print("\n== FIRST 5 DOCS (with embeddings) ==")
res = col.get(limit=5, include=["metadatas", "documents", "embeddings"])

ids = res.get("ids", [])

embs = res.get("embeddings", None)
# normalize to a Python list of vectors or None
if embs is None:
    embs = []
else:
    # if it's a numpy array, convert to list; otherwise leave as-is
    try:
        # check without importing numpy explicitly
        import numpy as _np  # noqa
        if hasattr(embs, "tolist"):
            embs = embs.tolist()
    except Exception:
        # best-effort: if it still behaves like an array, wrap in list()
        try:
            embs = list(embs)
        except Exception:
            pass

n = min(len(ids), len(embs)) if embs else len(ids)

for i in range(n):
    id_ = ids[i]
    emb = embs[i] if i < len(embs) else None
    dim = None if emb is None else len(emb)
    print(f"{i+1:>2}. {id_} | emb_dim={dim}")

# count how many are None (only for the portion we fetched)
none_count = sum(1 for i in range(min(len(ids), len(embs))) if embs[i] is None) if embs else 0
print("Embeddings None (from first batch):", none_count)

# Optional: detail of the first doc
if ids:
    got_one = col.get(ids=[ids[0]], include=["metadatas", "documents", "embeddings"])
    print("\n== DETAIL DOC PERTAMA ==")
    pprint({k: got_one[k] for k in got_one if k in ("ids","metadatas","documents")})
    e0 = got_one.get("embeddings", [None])[0]
    if hasattr(e0, "tolist"):
        e0 = e0.tolist()
    print("emb_dim (doc[0]):", None if e0 is None else len(e0))

## **DELETE COLLECTION**

In [ ]:
import chromadb

CHROMA_DIR = "./chroma_db"
COLLECTION_NAME = "classicmodels"

client = chromadb.PersistentClient(path=CHROMA_DIR)

# cek dulu ada atau nggak
try:
    client.get_collection(COLLECTION_NAME)  # kalau gak ada bakal raise
    client.delete_collection(COLLECTION_NAME)
    print(f"[OK] collection '{COLLECTION_NAME}' deleted.")
except Exception as e:
    print(f"[INFO] collection '{COLLECTION_NAME}' not found or already deleted. ({e})")